In [1]:
import os
import sys
import datetime
import pandas as pd
import time
import traceback
import requests

from bithumb_plug import InitBithumbAdaptor



In [2]:
bithumb_adaptor = InitBithumbAdaptor()

2024-02-13 11:07:38,902 - bithumb_plug - INFO - bithumb_plug_logger started.


In [3]:
result = bithumb_adaptor.wallet_status()
result

,asset,network_type,deposit,withdraw
0,BTC,BTC,True,True
0,ETH,ETH,True,True
1,ETH,ARB,True,True
2,ETH,OP,True,True
0,ETC,ETC,True,True
...,...,...,...,...
0,KWENTA,OP,True,True
0,LAZIO,BSC,True,True
0,PORTO,BSC,True,True
0,SANTOS,BSC,True,True


In [ ]:
network_df

,currency,net_type,deposit_status,withdrawal_status
0,C0101,BTC,1,1
0,C0102,ETH,1,1
1,C0102,ARB_ETH,1,1
2,C0102,OP_ETH,1,1
0,C0105,ETC,1,1
...,...,...,...,...
0,C0746,OP_ETH,1,1
0,C0747,BSC,1,1
0,C0749,BSC,1,1
0,C0760,BSC,1,1


In [48]:
res = requests.get('https://api.bithumb.com/public/withdraw/minimum/BTC')
res.json()['data']

[{'currency': 'BTC', 'net_type': 'BTC', 'minimum': '0.002'}]

In [12]:
res = requests.get('https://api.bithumb.com/public/network-info')
res.json()['data'][:5]

[{'net_type': 'BTC', 'net_name': 'Bitcoin'},
 {'net_type': 'ETH', 'net_name': 'Ethereum'},
 {'net_type': 'XRP', 'net_name': 'XRP'},
 {'net_type': 'TRX', 'net_name': 'Tron'},
 {'net_type': 'EOS', 'net_name': 'EOS'}]

In [41]:
res = requests.get('https://api.bithumb.com/public/assetsstatus/multichain/FCT2')
network_status_df = pd.DataFrame(res.json()['data'])
network_status_df

,currency,net_type,deposit_status,withdrawal_status
0,C0255,FCT2,1,1


In [38]:
network_status_df[network_status_df['net_type'].str.contains('ETH')]

,currency,net_type,deposit_status,withdrawal_status
1,C0102,ETH,1,1
2,C0102,ARB_ETH,1,1
3,C0102,OP_ETH,1,1
18,C0119,ETH,0,0
19,C0121,ETH,0,0
...,...,...,...,...
433,C0863,ETH,1,1
434,C0864,ARB_ETH,1,1
436,C0868,ETH,1,1
437,C0869,ETH,1,1


In [2]:
class Bithumb:
    def __init__(self, api_key=None, secret_key=None):
        self.api_key = api_key
        self.secret_key = secret_key
        self.server_url = "https://api.bithumb.com"
        self.headers = {"accept": "application/json"}
        self.chrome_driver_path = "/usr/local/bin/chromedriver"

    def spot_all_tickers(self, market=None):
        if market is None:
            market_list = ["KRW", "BTC"]
        else:
            market_list = [market]
        total_df = pd.DataFrame()
        for each_market in market_list:
            url = f"/public/ticker/ALL_{each_market}"
            res = requests.get(self.server_url + url, headers=self.headers)
            ticker_df = pd.DataFrame(res.json()['data']).T.reset_index()
            ticker_df['quote_asset'] = each_market
            total_df = pd.concat([total_df, ticker_df], axis=0)
        total_df.reset_index(drop=True, inplace=True)
        total_df = total_df[total_df['index'] != 'date'].rename(columns={'index': 'base_asset', 'acc_trade_value_24H':'atp24h', "closing_price":"lastPrice"})
        total_df.loc[:, 'opening_price':'fluctate_rate_24H'] = total_df.loc[:, 'opening_price':'fluctate_rate_24H'].astype(float)
        total_df['symbol'] = total_df['base_asset'] + '_' + total_df['quote_asset']
        return total_df
    
    def spot_exchange_info(self):
        return self.spot_all_tickers()

    def wallet_status(self):
        url = "https://www.bithumb.com/coin_inout/compare_price"
        # Set up driver
        options = webdriver.ChromeOptions()
        options.add_argument('--headless')
        options.add_argument('--no-sandbox')
        options.add_argument("user-agent=Mozilla/5.0 (Windows NT 6.1; WOW64; Trident/7.0; rv:11.0) like Gecko")
        # chrome_options.add_argument("--disable-dev-shm-usage")
        driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=options)

        # Get a webpage
        driver.get(url)
        wait = WebDriverWait(driver, 10)
        element = wait.until(EC.presence_of_element_located((By.CLASS_NAME, 'g_tb_list')))

        element_list = driver.find_elements(By.CLASS_NAME, 'coin_list')
        element_list = [x.text for x in element_list]

        # Clean up (close browser once done)
        driver.quit()

        wallet_status_list = [x.split('(')[1] for x in element_list]
        wallet_status_list = [x.replace(')','').split(' ') for x in wallet_status_list]
        wallet_status_df = pd.DataFrame(wallet_status_list)
        def temp(row):
            try:
                float(row[2])
                if row[1] == "Mainnet":
                    return row[0]
                else:
                    return row[1]
            except:
                network_name = row[1] + " " + row[2]
                return network_name.upper()
        wallet_status_df['network_type'] = wallet_status_df.apply(lambda row:temp(row), axis=1)
        wallet_status_df = wallet_status_df.rename(columns={0:'asset', 3: 'deposit', 4: 'withdraw'})
        wallet_status_df.loc[:, 'deposit'] = wallet_status_df['deposit'].apply(lambda x: True if x == '정상' else False)
        wallet_status_df.loc[:, 'withdraw'] = wallet_status_df['withdraw'].apply(lambda x: True if x == '정상' else False)
        wallet_status_df['network_type'] = wallet_status_df['network_type'].replace('ERC-20', 'ETH').replace('TRC-20', 'TRX').replace('BEP-20', 'BSC')
        wallet_status_df = wallet_status_df[['asset','network_type','deposit','withdraw']]

        return wallet_status_df